In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from google.colab import drive

# ----------------- 1. BAĞLANTI -----------------
print("🔄 Drive'a bağlanılıyor...")
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

TRAIN_CSV = '/content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv'
VAL_CSV   = '/content/drive/MyDrive/video_koordinat_verisetival2.csv'

assert os.path.exists(TRAIN_CSV) and os.path.exists(VAL_CSV)
print(f"✅ Train: {TRAIN_CSV}\n✅ Val: {VAL_CSV}")

# ----------------- 2. DOĞRULAMA -----------------
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header_line = f.readline().strip()
assert 'video_name' in header_line and 'class_id' in header_line
print("✅ CSV formatı doğru")

# ----------------- 3. VİDEO SAYISI -----------------
def count_videos(filepath):
    count, prev = 0, None
    with open(filepath, 'r', encoding='utf-8') as f:
        f.readline()
        for line in f:
            parts = line.strip().split(',')
            if len(parts) > 0 and parts[0] != prev:
                count += 1
                prev = parts[0]
    return count

print("\n📊 Video sayıları hesaplanıyor...")
train_count = count_videos(TRAIN_CSV)
val_count   = count_videos(VAL_CSV)
print(f"Train: {train_count} | Val: {val_count}")

# ----------------- 4. AYARLAR -----------------
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header = f.readline().strip().split(',')
feature_names = header[3:]

# Bacak noktalarını filtrele (pose 25+)
bad_indices = set()
for i, col_name in enumerate(feature_names):
    if 'pose_' in col_name:
        try:
            if int(col_name.split('_')[1]) >= 25:
                bad_indices.add(i)
        except: pass

valid_indices = [i for i in range(len(feature_names)) if i not in bad_indices]

def get_idx(name):
    try: return feature_names.index(name)
    except: return -1

# Burun
idx_nose_x = get_idx('pose_0_x')
idx_nose_y = get_idx('pose_0_y')

# Sağ el bilek
idx_rh_x = get_idx('rh_0_x')
idx_rh_y = get_idx('rh_0_y')

# Sol el bilek
idx_lh_x = get_idx('lh_0_x')
idx_lh_y = get_idx('lh_0_y')

# Ağız noktaları (pose_9, pose_10)
idx_mouth_x  = get_idx('pose_9_x')
idx_mouth_y  = get_idx('pose_9_y')

# Kaş/göz noktaları (pose_2 sağ göz, pose_5 sol göz)
idx_reye_x = get_idx('pose_2_x')
idx_reye_y = get_idx('pose_2_y')
idx_leye_x = get_idx('pose_5_x')
idx_leye_y = get_idx('pose_5_y')

# Omuz ortası (pose_11, pose_12)
idx_rs_x = get_idx('pose_11_x')
idx_rs_y = get_idx('pose_11_y')
idx_ls_x = get_idx('pose_12_x')
idx_ls_y = get_idx('pose_12_y')

# her el için 4 mesafe = 8 toplam + 2 el hız = 10
ADDED_FEATURES   = 10
BASE_FEATURES    = len(valid_indices)
NEW_NUM_FEATURES = BASE_FEATURES + ADDED_FEATURES

NUM_CLASSES  = 226
MAX_FRAMES   = 60
BATCH_SIZE   = 32
TRAIN_STEPS  = train_count // BATCH_SIZE
VAL_STEPS    = val_count   // BATCH_SIZE

print(f"\n📐 Baz özellik: {BASE_FEATURES} + Eklenen: {ADDED_FEATURES} = {NEW_NUM_FEATURES}")
print(f"📐 Train adım: {TRAIN_STEPS} | Val adım: {VAL_STEPS}")

# ----------------- 5. MESAFE HESABI -----------------
def dist(raw, ix, iy, jx, jy):
    """İki nokta arasındaki 2D mesafe"""
    if ix < 0 or iy < 0 or jx < 0 or jy < 0:
        return 0.0
    if ix >= len(raw) or jx >= len(raw):
        return 0.0
    return ((raw[ix]-raw[jx])**2 + (raw[iy]-raw[jy])**2)**0.5

# ----------------- 6. GENERATOR -----------------
def robust_generator(filepath):
    while True:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            f.readline()
            current_video, current_label, frames = None, None, []
            prev_rh_x, prev_rh_y = 0.0, 0.0
            prev_lh_x, prev_lh_y = 0.0, 0.0

            for line in f:
                parts = line.strip().split(',')
                try:
                    vid_name = parts[0]
                    label    = int(float(parts[1]))
                    raw      = [float(x) for x in parts[3:]]
                except:
                    continue

                # Burun merkezleme
                nx = raw[idx_nose_x] if idx_nose_x >= 0 and idx_nose_x < len(raw) else 0.0
                ny = raw[idx_nose_y] if idx_nose_y >= 0 and idx_nose_y < len(raw) else 0.0

                norm_feat = []
                for i in valid_indices:
                    val = raw[i] if i < len(raw) else 0.0
                    if feature_names[i].endswith('_x'): val -= nx
                    elif feature_names[i].endswith('_y'): val -= ny
                    norm_feat.append(val / 2.0)

                # --- YENİ: Mesafe özellikleri ---
                # Sağ el → ağız mesafesi
                d_rh_mouth = dist(raw, idx_rh_x, idx_rh_y, idx_mouth_x, idx_mouth_y)
                # Sağ el → sağ göz/kaş mesafesi
                d_rh_reye  = dist(raw, idx_rh_x, idx_rh_y, idx_reye_x, idx_reye_y)
                # Sağ el → burun mesafesi
                d_rh_nose  = dist(raw, idx_rh_x, idx_rh_y, idx_nose_x, idx_nose_y)
                # Sağ el → omuz ortası mesafesi
                d_rh_shldr = dist(raw, idx_rh_x, idx_rh_y, idx_rs_x, idx_rs_y)

                # Sol el → ağız mesafesi
                d_lh_mouth = dist(raw, idx_lh_x, idx_lh_y, idx_mouth_x, idx_mouth_y)
                # Sol el → sol göz/kaş mesafesi
                d_lh_leye  = dist(raw, idx_lh_x, idx_lh_y, idx_leye_x, idx_leye_y)
                # Sol el → burun mesafesi
                d_lh_nose  = dist(raw, idx_lh_x, idx_lh_y, idx_nose_x, idx_nose_y)
                # Sol el → omuz ortası mesafesi
                d_lh_shldr = dist(raw, idx_lh_x, idx_lh_y, idx_ls_x, idx_ls_y)

                # --- YENİ: El hızı (önceki kareden fark) ---
                cur_rh_x = raw[idx_rh_x] if idx_rh_x >= 0 and idx_rh_x < len(raw) else 0.0
                cur_rh_y = raw[idx_rh_y] if idx_rh_y >= 0 and idx_rh_y < len(raw) else 0.0
                cur_lh_x = raw[idx_lh_x] if idx_lh_x >= 0 and idx_lh_x < len(raw) else 0.0
                cur_lh_y = raw[idx_lh_y] if idx_lh_y >= 0 and idx_lh_y < len(raw) else 0.0

                vel_rh = ((cur_rh_x - prev_rh_x)**2 + (cur_rh_y - prev_rh_y)**2)**0.5
                vel_lh = ((cur_lh_x - prev_lh_x)**2 + (cur_lh_y - prev_lh_y)**2)**0.5

                prev_rh_x, prev_rh_y = cur_rh_x, cur_rh_y
                prev_lh_x, prev_lh_y = cur_lh_x, cur_lh_y

                extra = [
                    d_rh_mouth, d_rh_reye, d_rh_nose, d_rh_shldr,
                    d_lh_mouth, d_lh_leye, d_lh_nose, d_lh_shldr,
                    vel_rh, vel_lh
                ]
                norm_feat.extend(extra)

                # Boyut sabitle
                if len(norm_feat) > NEW_NUM_FEATURES:
                    norm_feat = norm_feat[:NEW_NUM_FEATURES]
                elif len(norm_feat) < NEW_NUM_FEATURES:
                    norm_feat.extend([0.0] * (NEW_NUM_FEATURES - len(norm_feat)))

                if current_video is None:
                    current_video, current_label = vid_name, label

                if vid_name != current_video:
                    if len(frames) > MAX_FRAMES:
                        frames = frames[:MAX_FRAMES]
                    else:
                        frames.extend([[0.0]*NEW_NUM_FEATURES]*(MAX_FRAMES-len(frames)))
                    yield (np.array(frames, dtype=np.float32),
                           tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))
                    current_video, current_label, frames = vid_name, label, [norm_feat]
                    prev_rh_x = prev_rh_y = prev_lh_x = prev_lh_y = 0.0
                else:
                    frames.append(norm_feat)

            if current_video is not None and len(frames) > 0:
                if len(frames) > MAX_FRAMES:
                    frames = frames[:MAX_FRAMES]
                else:
                    frames.extend([[0.0]*NEW_NUM_FEATURES]*(MAX_FRAMES-len(frames)))
                yield (np.array(frames, dtype=np.float32),
                       tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))

# ----------------- 7. DATASET -----------------
output_sig = (
    tf.TensorSpec(shape=(MAX_FRAMES, NEW_NUM_FEATURES), dtype=tf.float32),
    tf.TensorSpec(shape=(NUM_CLASSES,),                 dtype=tf.float32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: robust_generator(TRAIN_CSV), output_signature=output_sig
).shuffle(buffer_size=500).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: robust_generator(VAL_CSV), output_signature=output_sig
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ----------------- 8. BATCH KONTROLÜ -----------------
print("\n🧪 İlk batch kontrol ediliyor...")
for x, y in train_dataset.take(1):
    print(f"X shape: {x.shape} | Y shape: {y.shape}")
    print(f"X min/max: {x.numpy().min():.3f} / {x.numpy().max():.3f}")
    print(f"Label dağılımı (ilk 8): {tf.argmax(y, axis=1).numpy()[:8]}")
    nan_count = np.isnan(x.numpy()).sum()
    print(f"NaN sayısı: {nan_count} {'❌ SORUN VAR!' if nan_count > 0 else '✅'}")

# ----------------- 9. MODEL (sıfırdan) -----------------
model = Sequential([
    Input(shape=(MAX_FRAMES, NEW_NUM_FEATURES)),
    LSTM(256, return_sequences=True, activation='tanh'),
    BatchNormalization(),
    Dropout(0.4),
    LSTM(128, return_sequences=True, activation='tanh'),
    BatchNormalization(),
    Dropout(0.4),
    LSTM(64, return_sequences=False, activation='tanh'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ----------------- 10. EĞİTİM -----------------
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/isaret_dili_beyni_v2.h5',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

print(f"\n🚀 Yeni model eğitimi başlıyor! (mesafe + hız özellikleriyle)")
model.fit(
    train_dataset,
    steps_per_epoch=TRAIN_STEPS,
    validation_data=val_dataset,
    validation_steps=VAL_STEPS,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

🔄 Drive'a bağlanılıyor...
✅ Train: /content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv
✅ Val: /content/drive/MyDrive/video_koordinat_verisetival2.csv
✅ CSV formatı doğru

📊 Video sayıları hesaplanıyor...
Train: 28132 | Val: 4418

📐 Baz özellik: 201 + Eklenen: 10 = 211
📐 Train adım: 879 | Val adım: 138

🧪 İlk batch kontrol ediliyor...
X shape: (32, 60, 211) | Y shape: (32, 226)
X min/max: -0.579 / 1.112
Label dağılımı (ilk 8): [225 210  24   3 123 138  68  62]
NaN sayısı: 0 ✅


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 256)        │       479,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 128)        │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 226)            │        29,154 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 765,026 (2.92 MB)

 Trainable params: 764,130 (2.91 MB)

 Non-trainable params: 896 (3.50 KB)

... (24 outputs left out)


Epoch 33: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 459s 522ms/step - accuracy: 0.8025 - loss: 0.6341 - val_accuracy: 0.7692 - val_loss: 0.9877 - learning_rate: 5.0000e-04
Epoch 34/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 371ms/step - accuracy: 0.8354 - loss: 0.5187
Epoch 34: val_accuracy improved from 0.76925 to 0.77763, saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5



Epoch 34: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 453s 515ms/step - accuracy: 0.8156 - loss: 0.5897 - val_accuracy: 0.7776 - val_loss: 0.9709 - learning_rate: 5.0000e-04
Epoch 35/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step - accuracy: 0.8450 - loss: 0.4885
Epoch 35: val_accuracy did not improve from 0.77763
879/879 ━━━━━━━━━━━━━━━━━━━━ 446s 508ms/step - accuracy: 0.8219 - loss: 0.5617 - val_accuracy: 0.7643 - val_loss: 1.0511 - learning_rate: 5.0000e-04
Epoch 36/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step - accuracy: 0.8538 - loss: 0.4707
Epoch 36: val_accuracy did not improve from 0.77763
879/879 ━━━━━━━━━━━━━━━━━━━━ 498s 567ms/step - accuracy: 0.8321 - loss: 0.5417 - val_accuracy: 0.7640 - val_loss: 1.0757 - learning_rate: 5.0000e-04
Epoch 37/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.8541 - loss: 0.4643
Epoch 37: val_accuracy improved from 0.77763 to 0.78125, saving model to /content/drive/MyDrive/


Epoch 37: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 399s 455ms/step - accuracy: 0.8324 - loss: 0.5365 - val_accuracy: 0.7812 - val_loss: 0.9989 - learning_rate: 5.0000e-04
Epoch 38/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 369ms/step - accuracy: 0.8634 - loss: 0.4368
Epoch 38: val_accuracy did not improve from 0.78125
879/879 ━━━━━━━━━━━━━━━━━━━━ 446s 508ms/step - accuracy: 0.8388 - loss: 0.5106 - val_accuracy: 0.7717 - val_loss: 0.9943 - learning_rate: 5.0000e-04
Epoch 39/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step - accuracy: 0.8631 - loss: 0.4259
Epoch 39: val_accuracy did not improve from 0.78125

Epoch 39: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
879/879 ━━━━━━━━━━━━━━━━━━━━ 446s 508ms/step - accuracy: 0.8406 - loss: 0.5051 - val_accuracy: 0.7808 - val_loss: 1.0063 - learning_rate: 5.0000e-04
Epoch 40/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 378ms/step - accuracy: 0.8705 - loss: 0.4032
Epoch 40: val_accu


Epoch 40: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 461s 525ms/step - accuracy: 0.8563 - loss: 0.4529 - val_accuracy: 0.7892 - val_loss: 0.9845 - learning_rate: 2.5000e-04
Epoch 41/100
701/879 ━━━━━━━━━━━━━━━━━━━━ 1:02 351ms/step - accuracy: 0.8895 - loss: 0.3532

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from google.colab import drive

# ----------------- 1. BAĞLANTI -----------------
print("🔄 Drive'a bağlanılıyor...")
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

TRAIN_CSV = '/content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv'
VAL_CSV   = '/content/drive/MyDrive/video_koordinat_verisetival2.csv'

assert os.path.exists(TRAIN_CSV) and os.path.exists(VAL_CSV)
print(f"✅ Train: {TRAIN_CSV}\n✅ Val: {VAL_CSV}")

# ----------------- 2. DOĞRULAMA -----------------
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header_line = f.readline().strip()
assert 'video_name' in header_line and 'class_id' in header_line
print("✅ CSV formatı doğru")

# ----------------- 3. VİDEO SAYISI -----------------
def count_videos(filepath):
    count, prev = 0, None
    with open(filepath, 'r', encoding='utf-8') as f:
        f.readline()
        for line in f:
            parts = line.strip().split(',')
            if len(parts) > 0 and parts[0] != prev:
                count += 1
                prev = parts[0]
    return count

print("\n📊 Video sayıları hesaplanıyor...")
train_count = count_videos(TRAIN_CSV)
val_count   = count_videos(VAL_CSV)
print(f"Train: {train_count} | Val: {val_count}")

# ----------------- 4. AYARLAR -----------------
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header = f.readline().strip().split(',')
feature_names = header[3:]

bad_indices = set()
for i, col_name in enumerate(feature_names):
    if 'pose_' in col_name:
        try:
            if int(col_name.split('_')[1]) >= 25:
                bad_indices.add(i)
        except: pass

valid_indices = [i for i in range(len(feature_names)) if i not in bad_indices]

def get_idx(name):
    try: return feature_names.index(name)
    except: return -1

idx_nose_x   = get_idx('pose_0_x')
idx_nose_y   = get_idx('pose_0_y')
idx_rh_x     = get_idx('rh_0_x')
idx_rh_y     = get_idx('rh_0_y')
idx_lh_x     = get_idx('lh_0_x')
idx_lh_y     = get_idx('lh_0_y')
idx_mouth_x  = get_idx('pose_9_x')
idx_mouth_y  = get_idx('pose_9_y')
idx_reye_x   = get_idx('pose_2_x')
idx_reye_y   = get_idx('pose_2_y')
idx_leye_x   = get_idx('pose_5_x')
idx_leye_y   = get_idx('pose_5_y')
idx_rs_x     = get_idx('pose_11_x')
idx_rs_y     = get_idx('pose_11_y')
idx_ls_x     = get_idx('pose_12_x')
idx_ls_y     = get_idx('pose_12_y')

ADDED_FEATURES   = 10
BASE_FEATURES    = len(valid_indices)
NEW_NUM_FEATURES = BASE_FEATURES + ADDED_FEATURES

NUM_CLASSES  = 226
MAX_FRAMES   = 60
BATCH_SIZE   = 32
TRAIN_STEPS  = train_count // BATCH_SIZE
VAL_STEPS    = val_count   // BATCH_SIZE

print(f"\n📐 Özellik sayısı: {NEW_NUM_FEATURES}")
print(f"📐 Train adım: {TRAIN_STEPS} | Val adım: {VAL_STEPS}")

# ----------------- 5. MESAFE HESABI -----------------
def dist(raw, ix, iy, jx, jy):
    if ix < 0 or iy < 0 or jx < 0 or jy < 0: return 0.0
    if ix >= len(raw) or jx >= len(raw): return 0.0
    return ((raw[ix]-raw[jx])**2 + (raw[iy]-raw[jy])**2)**0.5

# ----------------- 6. GENERATOR -----------------
def robust_generator(filepath):
    while True:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            f.readline()
            current_video, current_label, frames = None, None, []
            prev_rh_x, prev_rh_y = 0.0, 0.0
            prev_lh_x, prev_lh_y = 0.0, 0.0

            for line in f:
                parts = line.strip().split(',')
                try:
                    vid_name = parts[0]
                    label    = int(float(parts[1]))
                    raw      = [float(x) for x in parts[3:]]
                except:
                    continue

                nx = raw[idx_nose_x] if idx_nose_x >= 0 and idx_nose_x < len(raw) else 0.0
                ny = raw[idx_nose_y] if idx_nose_y >= 0 and idx_nose_y < len(raw) else 0.0

                norm_feat = []
                for i in valid_indices:
                    val = raw[i] if i < len(raw) else 0.0
                    if feature_names[i].endswith('_x'): val -= nx
                    elif feature_names[i].endswith('_y'): val -= ny
                    norm_feat.append(val / 2.0)

                d_rh_mouth = dist(raw, idx_rh_x, idx_rh_y, idx_mouth_x, idx_mouth_y)
                d_rh_reye  = dist(raw, idx_rh_x, idx_rh_y, idx_reye_x,  idx_reye_y)
                d_rh_nose  = dist(raw, idx_rh_x, idx_rh_y, idx_nose_x,  idx_nose_y)
                d_rh_shldr = dist(raw, idx_rh_x, idx_rh_y, idx_rs_x,    idx_rs_y)
                d_lh_mouth = dist(raw, idx_lh_x, idx_lh_y, idx_mouth_x, idx_mouth_y)
                d_lh_leye  = dist(raw, idx_lh_x, idx_lh_y, idx_leye_x,  idx_leye_y)
                d_lh_nose  = dist(raw, idx_lh_x, idx_lh_y, idx_nose_x,  idx_nose_y)
                d_lh_shldr = dist(raw, idx_lh_x, idx_lh_y, idx_ls_x,    idx_ls_y)

                cur_rh_x = raw[idx_rh_x] if idx_rh_x >= 0 and idx_rh_x < len(raw) else 0.0
                cur_rh_y = raw[idx_rh_y] if idx_rh_y >= 0 and idx_rh_y < len(raw) else 0.0
                cur_lh_x = raw[idx_lh_x] if idx_lh_x >= 0 and idx_lh_x < len(raw) else 0.0
                cur_lh_y = raw[idx_lh_y] if idx_lh_y >= 0 and idx_lh_y < len(raw) else 0.0

                vel_rh = ((cur_rh_x-prev_rh_x)**2 + (cur_rh_y-prev_rh_y)**2)**0.5
                vel_lh = ((cur_lh_x-prev_lh_x)**2 + (cur_lh_y-prev_lh_y)**2)**0.5

                prev_rh_x, prev_rh_y = cur_rh_x, cur_rh_y
                prev_lh_x, prev_lh_y = cur_lh_x, cur_lh_y

                norm_feat.extend([
                    d_rh_mouth, d_rh_reye, d_rh_nose, d_rh_shldr,
                    d_lh_mouth, d_lh_leye, d_lh_nose, d_lh_shldr,
                    vel_rh, vel_lh
                ])

                if len(norm_feat) > NEW_NUM_FEATURES:
                    norm_feat = norm_feat[:NEW_NUM_FEATURES]
                elif len(norm_feat) < NEW_NUM_FEATURES:
                    norm_feat.extend([0.0] * (NEW_NUM_FEATURES - len(norm_feat)))

                if current_video is None:
                    current_video, current_label = vid_name, label

                if vid_name != current_video:
                    if len(frames) > MAX_FRAMES:
                        frames = frames[:MAX_FRAMES]
                    else:
                        frames.extend([[0.0]*NEW_NUM_FEATURES]*(MAX_FRAMES-len(frames)))
                    yield (np.array(frames, dtype=np.float32),
                           tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))
                    current_video, current_label, frames = vid_name, label, [norm_feat]
                    prev_rh_x = prev_rh_y = prev_lh_x = prev_lh_y = 0.0
                else:
                    frames.append(norm_feat)

            if current_video is not None and len(frames) > 0:
                if len(frames) > MAX_FRAMES:
                    frames = frames[:MAX_FRAMES]
                else:
                    frames.extend([[0.0]*NEW_NUM_FEATURES]*(MAX_FRAMES-len(frames)))
                yield (np.array(frames, dtype=np.float32),
                       tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))

# ----------------- 7. DATASET -----------------
output_sig = (
    tf.TensorSpec(shape=(MAX_FRAMES, NEW_NUM_FEATURES), dtype=tf.float32),
    tf.TensorSpec(shape=(NUM_CLASSES,),                 dtype=tf.float32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: robust_generator(TRAIN_CSV), output_signature=output_sig
).shuffle(buffer_size=500).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: robust_generator(VAL_CSV), output_signature=output_sig
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ----------------- 8. BATCH KONTROLÜ -----------------
print("\n🧪 İlk batch kontrol ediliyor...")
for x, y in train_dataset.take(1):
    print(f"X shape: {x.shape} | Y shape: {y.shape}")
    print(f"X min/max: {x.numpy().min():.3f} / {x.numpy().max():.3f}")
    nan_count = np.isnan(x.numpy()).sum()
    print(f"NaN sayısı: {nan_count} {'❌' if nan_count > 0 else '✅'}")

# ----------------- 9. MODELİ YÜKLE -----------------
CHECKPOINT_PATH = '/content/drive/MyDrive/isaret_dili_beyni_v2.h5'

print(f"\n✅ v2 modeli yükleniyor (epoch 40, val_acc ~%78.9)...")
model = tf.keras.models.load_model(CHECKPOINT_PATH)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00025),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("📐 Learning rate: 0.00025 (epoch 40 seviyesi)")
model.summary()

# ----------------- 10. EĞİTİM -----------------
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/isaret_dili_beyni_v2.h5',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

print(f"\n🚀 Epoch 40'tan devam ediliyor!")
model.fit(
    train_dataset,
    steps_per_epoch=TRAIN_STEPS,
    validation_data=val_dataset,
    validation_steps=VAL_STEPS,
    epochs=100,
    initial_epoch=40,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

🔄 Drive'a bağlanılıyor...
Mounted at /content/drive
✅ Train: /content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv
✅ Val: /content/drive/MyDrive/video_koordinat_verisetival2.csv
✅ CSV formatı doğru

📊 Video sayıları hesaplanıyor...
Train: 28132 | Val: 4418

📐 Özellik sayısı: 211
📐 Train adım: 879 | Val adım: 138

🧪 İlk batch kontrol ediliyor...
X shape: (32, 60, 211) | Y shape: (32, 226)
X min/max: -0.621 / 1.107
NaN sayısı: 0 ✅

✅ v2 modeli yükleniyor (epoch 40, val_acc ~%78.9)...


📐 Learning rate: 0.00025 (epoch 40 seviyesi)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 256)        │       479,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 128)        │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 226)            │        29,154 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 765,026 (2.92 MB)

 Trainable params: 764,130 (2.91 MB)

... (6 outputs left out)


Epoch 49: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 440s 501ms/step - accuracy: 0.8912 - loss: 0.3332 - val_accuracy: 0.8053 - val_loss: 1.0324 - learning_rate: 1.2500e-04
Epoch 50/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - accuracy: 0.9074 - loss: 0.2884
Epoch 50: val_accuracy improved from 0.80525 to 0.80729, saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5



Epoch 50: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 437s 497ms/step - accuracy: 0.8934 - loss: 0.3319 - val_accuracy: 0.8073 - val_loss: 1.0058 - learning_rate: 1.2500e-04
Epoch 51/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step - accuracy: 0.9073 - loss: 0.2836
Epoch 51: val_accuracy did not improve from 0.80729

Epoch 51: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.
879/879 ━━━━━━━━━━━━━━━━━━━━ 435s 496ms/step - accuracy: 0.8919 - loss: 0.3294 - val_accuracy: 0.8037 - val_loss: 1.0411 - learning_rate: 1.2500e-04
Epoch 52/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 360ms/step - accuracy: 0.9078 - loss: 0.2843
Epoch 52: val_accuracy did not improve from 0.80729
879/879 ━━━━━━━━━━━━━━━━━━━━ 435s 495ms/step - accuracy: 0.8960 - loss: 0.3213 - val_accuracy: 0.8066 - val_loss: 1.0337 - learning_rate: 6.2500e-05
Epoch 53/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 364ms/step - accuracy: 0.9102 - loss: 0.2765
Epoch 53: val_accur


Epoch 53: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_v2.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 443s 504ms/step - accuracy: 0.8994 - loss: 0.3107 - val_accuracy: 0.8118 - val_loss: 1.0198 - learning_rate: 6.2500e-05
Epoch 54/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step - accuracy: 0.9097 - loss: 0.2811
Epoch 54: val_accuracy did not improve from 0.81182
879/879 ━━━━━━━━━━━━━━━━━━━━ 430s 490ms/step - accuracy: 0.8990 - loss: 0.3143 - val_accuracy: 0.8068 - val_loss: 1.0323 - learning_rate: 6.2500e-05
Epoch 55/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step - accuracy: 0.9154 - loss: 0.2621
Epoch 55: val_accuracy did not improve from 0.81182
879/879 ━━━━━━━━━━━━━━━━━━━━ 439s 500ms/step - accuracy: 0.9031 - loss: 0.3022 - val_accuracy: 0.8098 - val_loss: 1.0272 - learning_rate: 6.2500e-05
Epoch 56/100
176/879 ━━━━━━━━━━━━━━━━━━━━ 1:41 144ms/step - accuracy: 0.9171 - loss: 0.2517

KeyboardInterrupt: 

In [ ]:
import tensorflow as tf

# H5 modelini yükle
model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')

# TFLite'a çevir
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # boyutu küçültür
tflite_model = converter.convert()

# Kaydet
with open('/content/drive/MyDrive/isaret_dili_beyni_v2.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ Tamamlandı!")
print(f"Boyut: {len(tflite_model) / 1024 / 1024:.2f} MB")

Saved artifact at '/tmp/tmp4363ojnd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 211), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 226), dtype=tf.float32, name=None)
Captures:
  132963561595792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561597136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561598480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561597712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561597904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561596944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561596752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561600016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561600208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561602128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132963561601936: 

ConverterError: Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(fused["CudnnRNNV3:", "sequential_1/lstm_1/CudnnRNNV3@__inference_function_1135"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_1246"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))): 'tf.CudnnRNNV3' op is neither a custom op nor a flex op
<unknown>:0: note: loc(callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_1246"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): called from
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: note: loc(callsite(fused["CudnnRNNV3:", "sequential_1/lstm_1/CudnnRNNV3@__inference_function_1135"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_1246"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))

In [ ]:
import tensorflow as tf
import numpy as np

# 1. Eski modelin ağırlıklarını yükle
old_model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')

# 2. Aynı mimariyle CPU uyumlu yeni model oluştur
# (implementation=1 → CPU, implementation=2 → GPU/CuDNN)
new_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(60, 211)),
    tf.keras.layers.LSTM(256, return_sequences=True, activation='tanh', implementation=1),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.LSTM(128, return_sequences=True, activation='tanh', implementation=1),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.LSTM(64, return_sequences=False, activation='tanh', implementation=1),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(226, activation='softmax')
])

# 3. Eski ağırlıkları yeni modele aktar
new_model.set_weights(old_model.get_weights())
print("✅ Ağırlıklar aktarıldı")

# 4. Doğruluk kontrolü — tahminler aynı mı?
test_input = np.random.rand(1, 60, 211).astype(np.float32)
old_pred = old_model.predict(test_input, verbose=0)
new_pred = new_model.predict(test_input, verbose=0)
print(f"Tahmin farkı: {np.max(np.abs(old_pred - new_pred)):.6f} (0'a yakın olmalı)")

# 5. TFLite'a çevir
converter = tf.lite.TFLiteConverter.from_keras_model(new_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# 6. Kaydet
with open('/content/drive/MyDrive/isaret_dili_beyni_v2.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ Tamamlandı!")
print(f"Boyut: {len(tflite_model) / 1024 / 1024:.2f} MB")

✅ Ağırlıklar aktarıldı
Tahmin farkı: 0.000000 (0'a yakın olmalı)
Saved artifact at '/tmp/tmpbxzsbu6q'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 211), dtype=tf.float32, name='keras_tensor_78')
Output Type:
  TensorSpec(shape=(None, 226), dtype=tf.float32, name=None)
Captures:
  138270264106000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264104272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264107920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264107728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264106768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264106960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264107344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264108112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264108496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138270264109264: T

ConverterError: Could not translate MLIR to FlatBuffer.<unknown>:0: error: loc(callsite(fused["CudnnRNNV3:", "sequential_1/lstm_1/CudnnRNNV3@__inference_function_2347"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_2458"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))): 'tf.CudnnRNNV3' op is neither a custom op nor a flex op
<unknown>:0: note: loc(callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_2458"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): called from
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: note: loc(callsite(fused["CudnnRNNV3:", "sequential_1/lstm_1/CudnnRNNV3@__inference_function_2347"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_2458"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))

In [ ]:
import tensorflow as tf
import numpy as np
import os

# GPU'yu tamamen kapat
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.config.set_visible_devices([], 'GPU')
print("✅ GPU kapatıldı, CPU modunda çalışıyor")

# Eski ağırlıkları numpy olarak kaydet
old_model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')
weights = old_model.get_weights()
print(f"✅ {len(weights)} ağırlık katmanı alındı")
del old_model  # belleği boşalt

# CPU'da yeni model oluştur
with tf.device('/CPU:0'):
    new_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(60, 211)),
        tf.keras.layers.LSTM(256, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(64, return_sequences=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(226, activation='softmax')
    ])
    new_model.set_weights(weights)
    print("✅ Ağırlıklar CPU modeline aktarıldı")

    # Doğrulama
    test = np.random.rand(1, 60, 211).astype(np.float32)
    pred = new_model.predict(test, verbose=0)
    print(f"✅ Test tahmini çalıştı: shape={pred.shape}")

# TFLite dönüşümü
converter = tf.lite.TFLiteConverter.from_keras_model(new_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

with open('/content/drive/MyDrive/isaret_dili_beyni_v2.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"\n✅ Tamamlandı!")
print(f"Boyut: {len(tflite_model) / 1024 / 1024:.2f} MB")

✅ GPU kapatıldı, CPU modunda çalışıyor


✅ 25 ağırlık katmanı alındı
✅ Ağırlıklar CPU modeline aktarıldı
✅ Test tahmini çalıştı: shape=(1, 226)
Saved artifact at '/tmp/tmp1hq93hei'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 211), dtype=tf.float32, name='keras_tensor_78')
Output Type:
  TensorSpec(shape=(None, 226), dtype=tf.float32, name=None)
Captures:
  140426616313744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616310288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616315664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616315472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616314512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616314704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616315088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616315856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426616316240: TensorSpec(shape=(), dtype=tf.res

ConverterError: <unknown>:0: error: loc(callsite(fused["TensorListReserve:", "sequential_1/lstm_1/TensorArrayV2_1@__inference_function_2272"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_2383"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]))): 'tf.TensorListReserve' op requires element_shape to be static during TF Lite transformation pass
<unknown>:0: note: loc(callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_2383"] at fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"])): called from
<unknown>:0: note: loc(fused["StatefulPartitionedCall:", "StatefulPartitionedCall_1"]): called from
<unknown>:0: error: loc(callsite(fused["TensorListReserve:", "sequential_1/lstm_1/TensorArrayV2_1@__inference_function_2272"] at callsite(fused["StatefulPartitionedCall:", "StatefulPartitionedCall@__inference_signature_wrapper_2383"] at fused["StatefulPartitionedCall:", "St

In [ ]:
import tensorflow as tf
import numpy as np
import os

# GPU'yu tamamen kapat
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.config.set_visible_devices([], 'GPU')
print("✅ GPU kapatıldı, CPU modunda çalışıyor")

# Eski ağırlıkları numpy olarak kaydet
old_model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')
weights = old_model.get_weights()
print(f"✅ {len(weights)} ağırlık katmanı alındı")
del old_model  # belleği boşalt

# CPU'da yeni model oluştur
with tf.device('/CPU:0'):
    new_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(60, 211)),
        tf.keras.layers.LSTM(256, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(64, return_sequences=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(226, activation='softmax')
    ])
    new_model.set_weights(weights)
    print("✅ Ağırlıklar CPU modeline aktarıldı")

    # Doğrulama
    test = np.random.rand(1, 60, 211).astype(np.float32)
    pred = new_model.predict(test, verbose=0)
    print(f"✅ Test tahmini çalıştı: shape={pred.shape}")

# --- YENİ VE GÜÇLENDİRİLMİŞ TFLITE DÖNÜŞÜMÜ ---
print("\n--- Son Dönüşüm Başladı (Lütfen Bekleyin) ---")
converter = tf.lite.TFLiteConverter.from_keras_model(new_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Hatanın bizden istediği iki özel ayar:
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

# Dönüştür ve Kaydet
tflite_model = converter.convert()

with open('/content/drive/MyDrive/isaret_dili_beyni_v2.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"\n✅ VE MUTLU SON! Tamamlandı!")
print(f"Boyut: {len(tflite_model) / 1024 / 1024:.2f} MB")

✅ GPU kapatıldı, CPU modunda çalışıyor


✅ 25 ağırlık katmanı alındı
✅ Ağırlıklar CPU modeline aktarıldı
✅ Test tahmini çalıştı: shape=(1, 226)

--- Son Dönüşüm Başladı (Lütfen Bekleyin) ---
Saved artifact at '/tmp/tmpjpc5kv5q'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 211), dtype=tf.float32, name='keras_tensor_169')
Output Type:
  TensorSpec(shape=(None, 226), dtype=tf.float32, name=None)
Captures:
  140426393603152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426509646544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426393604880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426393605264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426393604304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426393602576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426393605072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140426393605456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14

In [ ]:
!pip install tensorflow==2.16.1

In [ ]:
import tensorflow as tf
import numpy as np
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
tf.config.set_visible_devices([], 'GPU')
print("✅ GPU kapatıldı, CPU modunda çalışıyor")

old_model = tf.keras.models.load_model('/content/drive/MyDrive/isaret_dili_beyni_v2.h5')
weights = old_model.get_weights()
print(f"✅ {len(weights)} ağırlık katmanı alındı")
del old_model

with tf.device('/CPU:0'):
    new_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(60, 211)),
        tf.keras.layers.LSTM(256, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(64, return_sequences=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(226, activation='softmax')
    ])
    new_model.set_weights(weights)

    test = np.random.rand(1, 60, 211).astype(np.float32)
    pred = new_model.predict(test, verbose=0)
    print(f"✅ Test tahmini çalıştı: shape={pred.shape}")

print("\n--- Dönüşüm Başladı ---")
converter = tf.lite.TFLiteConverter.from_keras_model(new_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open('/content/drive/MyDrive/isaret_dili_beyni_v2.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ Tamamlandı! Boyut: {len(tflite_model) / 1024 / 1024:.2f} MB")

ValueError: JAX requires ml_dtypes version 0.5 or newer; installed version is 0.3.2.